# ERM 424 · Module 5 · Stage 1: Framing and Quantifying

The practice data is behind you. This week you are handed a client: Lakefill National Bank, a mid-size commercial bank whose Chief Risk Officer (CRO) has engaged you to assess its operational loss exposure ahead of capital planning. The CRO's letter sets three questions; Stage 1 answers none of them yet and makes all of them answerable.

- **Part A, the engagement opens** (about 3 hours, 63 points): your charter on the provided template, the prep summary as a decision table, the fitness memo against BCBS 239 Principles 3 to 6 (BCBS 239 is the Basel Committee on Banking Supervision's 2013 standard, *Principles for effective risk data aggregation and risk reporting*), the frequency and severity candidates for your business line, and the change line.
- **Part B, audit the AI's opening** (about 1 hour, 27 points; AI means artificial intelligence): your assistant drafts charter language and proposes frequency and severity candidates; you disposition each element against your own fitness findings. The audit is graded, not the AI's output.

**Before anything else: File, then Save a copy in Drive.** Work in your copy.

Course objectives served: CO1, CO3, CO5, CO6.

Version 1.0, 2026-11-11. Read alongside: [the CRO sponsor letter](https://github.com/nybabot/ERM424/blob/main/docs/module_05/cro_sponsor_letter.md), [the data dictionary](https://github.com/nybabot/ERM424/blob/main/data/lakefill/lakefill_DICTIONARY.md) and [the Basel event-type reference sheet](https://github.com/nybabot/ERM424/blob/main/docs/module_05/basel_event_type_reference.md), all in the course repository.

In [ ]:
# Setup: run this cell first.
import sys, warnings
import numpy as np
import pandas as pd
EXPECTED = {"pandas": "2.2", "numpy": "2.4"}
for name, want in EXPECTED.items():
    have = __import__(name).__version__
    if not have.startswith(want):
        print(f"Note: {name} is {have}; this notebook was built on {want}.x. Most cells will still run.")
import os, time
import matplotlib.pyplot as plt
# Chart colors: the Okabe-Ito palette, readable with most forms of color-vision difference. Meaning never rides on color alone.
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=OKABE_ITO)
plt.rcParams["figure.dpi"] = 100

# Where the course files live. By default (Colab) the notebook reads them from the course repository on GitHub.
# If the course folders sit next to this notebook (a local copy of the repository), LOCAL becomes True and it reads them from disk.
REPO = "nybabot/ERM424"   # course repository on GitHub, as owner/name
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/claims/"
GENERATOR_URL = f"https://raw.githubusercontent.com/{REPO}/main/generators/claims_world.py"
LOCAL = False
DATA_DIR = "../../data/claims"   # notebooks/solutions, notebooks/student and docs/sync sit two folders below the course root
GEN_DIR = "../../generators"
for _cand in ["../../", "../", "./", "/content/"]:   # also works from qa/, from the root, or on a Colab VM with the folders under /content
    if os.path.isdir(_cand + "data/claims"):
        DATA_DIR, GEN_DIR = _cand + "data/claims", _cand + "generators"
        LOCAL = True
        break

def load(name, **kwargs):
    """Read one course CSV from the local data folder (LOCAL) or from the course repository (published)."""
    source = os.path.join(DATA_DIR, name) if LOCAL else BASE_URL + name
    return pd.read_csv(source, **kwargs)

print("pandas", pd.__version__, "| numpy", np.__version__, "| reading data from", os.path.abspath(DATA_DIR) if LOCAL else BASE_URL)

# Module 5 onward: the Lakefill files sit beside the claims files, in data/lakefill/ of the same repository.
from scipy import stats
LAKEFILL_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/lakefill/"
LAKEFILL_DIR = os.path.join(os.path.dirname(DATA_DIR), "lakefill")

def load(name, **kwargs):
    """Read one Lakefill CSV from the local data folder (LOCAL) or from the course repository (published)."""
    source = os.path.join(LAKEFILL_DIR, name) if LOCAL else LAKEFILL_URL + name
    return pd.read_csv(source, **kwargs)

print("Lakefill files from", os.path.abspath(LAKEFILL_DIR) if LOCAL else LAKEFILL_URL)


## Part A: The engagement opens

About 3 hours. Your **charter** is written on [the engagement charter template](https://github.com/nybabot/ERM424/blob/main/docs/module_05/engagement_charter_template.md) and submitted beside this notebook as its own file; [the worked example for a hospital system](https://github.com/nybabot/ERM424/blob/main/docs/module_05/worked_charter_example_hospital.md) shows the genre for a different sponsor. Write the charter's purpose sentence before Step A4, because the fitness memo judges the data against it.

This notebook holds everything else Part A asks for: the prep summary (Step A1), the frequency call (Step A2), the severity comparison and carry-forward choice (Step A3), the fitness memo (Step A4) and the change line (Step A5). Analytical results within a desk are expected to converge; grading weight sits on judgment: your rationales, your fitness reasoning, and the audit.

### Step A0: Load the register and choose your line

Two files, both described in [the data dictionary](https://github.com/nybabot/ERM424/blob/main/data/lakefill/lakefill_DICTIONARY.md). The register has one row per operational loss event, 2021 to 2025, as extracted on 2026-03-31; the exposure table has one row per business line per month. Blank means not recorded. Nothing in this notebook edits the files: your preparation in Step A1 works on a copy in memory.

In [ ]:
# Load the raw register with its four date columns read as dates, and show its shape and first rows.
raw = load("lakefill_losses_raw.csv", parse_dates=["occurrence_date", "discovery_date", "reporting_date", "close_date"])
print(raw.shape)
raw.head()

In [ ]:
# Load the monthly exposure table: one row per business line per month.
exposure = load("lakefill_exposure_monthly.csv")
print(exposure.shape)
exposure.head()

Set `LINE` to your desk's business line, spelled exactly as in the file. Every per-line cell below reads it, so this is the only edit the notebook needs. The bank-wide checks in Step A1 run on every line, so you see your line against the bank.

In [ ]:
# Your desk's business line: replace the text in quotes with your line, spelled as in the file.
LINE = "Retail Banking"

In [ ]:
# Given: confirm LINE is one of the six lines, and show its row count and first and last discovery dates.
_lines = sorted(raw["business_line"].unique())
if LINE not in _lines:
    print(f"LINE is {LINE!r}, which is not a business line in the file. Choose one of: {_lines}")
else:
    _rows = raw[raw["business_line"] == LINE]
    print(f"{LINE}: {len(_rows):,} rows, discovered {_rows['discovery_date'].min():%Y-%m-%d} to {_rows['discovery_date'].max():%Y-%m-%d}")

### Step A1: Look before you fit

The file is raw and the preparation is light. Six checks follow, each on the whole bank by line. They do not tell you what is wrong; they show you where to look. For anything that stands out, read the dictionary before you decide: some patterns are errors, and some are facts about how the bank collected its data. What you find here feeds the fitness memo (Step A4) and the prep summary.

In [ ]:
# Given: check 1, blanks. Columns with any blank, counted by line; then blank event_type by line and year, and blank recoveries on Closed rows by line and year.
_blank_cols = raw.columns[raw.isna().any()]
display(raw[_blank_cols].isna().groupby(raw["business_line"]).sum())
_year = raw["discovery_date"].dt.year.rename("year")
print("blank event_type, by line and discovery year")
display(raw["event_type"].isna().groupby([raw["business_line"], _year]).sum().unstack(fill_value=0))
_closed = raw[raw["status"] == "Closed"]
print("blank recoveries on Closed rows, by line and discovery year")
display(_closed["recoveries"].isna().groupby([_closed["business_line"], _closed["discovery_date"].dt.year.rename("year")]).sum().unstack(fill_value=0))

In [ ]:
# Given: check 2, repeated rows. Rows identical on every column except event_id and reporting_date, counted by line and discovery quarter (each such row and its partner are both counted).
_compare = [col for col in raw.columns if col not in ("event_id", "reporting_date")]
_repeat = raw.duplicated(subset=_compare, keep=False)
print("rows that have an identical partner:", int(_repeat.sum()))
if _repeat.any():
    _r = raw[_repeat]
    display(_r.groupby([_r["business_line"], _r["discovery_date"].dt.to_period("Q").rename("quarter")]).size().rename("rows").to_frame())

If check 2 shows rows for your line, decide whether they are one event logged twice or two genuine events that happen to match, and say what evidence decides it (the dictionary, the quarter's count against its neighbors, the `reporting_date` gap). Either answer is a row in your prep summary.

In [ ]:
# Given: check 3, date order and reporting lag. Occurrence after discovery and reporting before discovery, by line; then the median reporting lag in days by line and year.
_order = pd.DataFrame({
    "occurrence after discovery": (raw["occurrence_date"] > raw["discovery_date"]).groupby(raw["business_line"]).sum(),
    "reporting before discovery": (raw["reporting_date"] < raw["discovery_date"]).groupby(raw["business_line"]).sum(),
})
display(_order)
_lag = (raw["reporting_date"] - raw["discovery_date"]).dt.days
print("median reporting lag (days), by line and discovery year")
display(_lag.groupby([raw["business_line"], raw["discovery_date"].dt.year.rename("year")]).median().unstack())

In [ ]:
# Given: check 4, status. Open and Closed rows by line; then the share of open events by discovery half-year.
display(raw.groupby("business_line")["status"].value_counts().unstack(fill_value=0))
_half = raw["discovery_date"].dt.year.astype(str) + " H" + ((raw["discovery_date"].dt.month > 6) + 1).astype(str)
print("share of events still open, by discovery half-year")
display((raw["status"] == "Open").groupby(_half).mean().round(3).rename("open share").to_frame().T)

In [ ]:
# Given: check 5, the floor. The smallest gross_loss by line and threshold_era, beside the collection thresholds the dictionary states.
display(raw.groupby(["business_line", "threshold_era"])["gross_loss"].min().unstack())
print("Dictionary: every line is recorded at 10,000 or more, except shared services, recorded at 20,000 or more before 2023-01-01.")
print("An open event's gross_loss is a current estimate (dictionary, status).")

In [ ]:
# Given: check 6, counts by line and discovery year (every status). Look at where each line starts and where a count steps.
display(raw.groupby([raw["business_line"], raw["discovery_date"].dt.year.rename("year")]).size().unstack(fill_value=0))

Now apply your light preparation. Build `ev` from `raw` with only the treatments you can defend; leave everything else alone and say so in the prep summary. Every cell from Step A2 onward runs on `ev`, so what you do here shows up in your numbers. Keep it light: blanks that reflect how the bank collected its data are usually documented, not filled; Stage 2 tests the handlings.

In [ ]:
# Your light preparation: build ev from raw, applying only treatments you can defend; record each one in the prep summary below.
ev = raw.copy()
# YOUR CODE HERE

In [ ]:
# Given: what your preparation changed. Row counts before and after, and checks 2 and 3 recomputed on ev.
_compare_ev = [col for col in ev.columns if col not in ("event_id", "reporting_date")]
print(f"raw: {len(raw):,} rows | ev: {len(ev):,} rows")
print("rows in ev with an identical partner:", int(ev.duplicated(subset=_compare_ev, keep=False).sum()))
print("rows in ev with occurrence after discovery:", int((ev["occurrence_date"] > ev["discovery_date"]).sum()))
print(f"{LINE}: {int((raw['business_line'] == LINE).sum()):,} rows in raw, {int((ev['business_line'] == LINE).sum()):,} in ev")

**The prep summary.** A decision table, one row per issue: what you did, why, and what you left alone. A check that found something you chose not to treat is a row too (treatment: "left as is", with the reason). Six columns, the same as the Module 3 decision log: issue, columns affected, mechanism and evidence, treatment, alternatives rejected, information lost.

Add a row with `log_decision(...)` and call `show_log()` to see the table. An example on an invented issue:

```python
log_decision(
    issue="Region typed two ways in an invented table",
    columns="region",
    mechanism_and_evidence="value_counts() shows 'Chicago Metro' and 'Chicago metro' as separate labels",
    treatment="Replaced the lower-case spelling on 4 rows",
    alternatives_rejected="Drop the 4 rows (rejected: the rest of each row is valid)",
    information_lost="None: a spelling variant")
```

In [ ]:
# Decision table template: log_decision(...) adds one row; show_log() displays the table so far.
LOG_COLUMNS = ["issue", "columns affected", "mechanism and evidence", "treatment", "alternatives rejected", "information lost"]
decision_log = []

def log_decision(issue, columns, mechanism_and_evidence, treatment, alternatives_rejected, information_lost):
    """Add one row to the prep summary."""
    values = [issue, columns, mechanism_and_evidence, treatment, alternatives_rejected, information_lost]
    decision_log.append(dict(zip(LOG_COLUMNS, values)))

def show_log():
    """Display the prep summary as a table."""
    if not decision_log:
        print("The log has no rows yet. Add one with log_decision(...).")
        return
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(decision_log))

In [ ]:
# Your prep summary rows: one log_decision(...) call per issue, for your line and for any bank-wide treatment.
decision_log.clear()   # clears the table first, so running this cell again does not add every row twice
# YOUR CODE HERE

In [ ]:
# Given: check the prep summary. It lists blank fields and never stops the notebook.
if not decision_log:
    print("No rows to check yet.")
else:
    def _blank(value):
        """True for a missing entry: None, NaN, or text that is only spaces."""
        return value is None or (isinstance(value, float) and value != value) or not str(value).strip()
    print(len(decision_log), "row(s) in the prep summary.")
    blanks = [(i + 1, col) for i, row in enumerate(decision_log) for col in LOG_COLUMNS if _blank(row[col])]
    if blanks:
        for number, col in blanks:
            print(f"Row {number} has no entry in: {col}")
    else:
        print("Every field of every logged row is filled in.")

### Step A2: How often do losses arrive?

Counts use `discovery_date` and every status: an open event is an event; only its amount is provisional. The first question is whether a Poisson model, whose variance equals its mean, survives your line's data. A variance-to-mean ratio near 1 is consistent with Poisson; a ratio well above 1 is overdispersion, which a negative binomial allows. A ratio above 1 can also come from a rate that changed between periods, which is not clustering, so the check runs on two windows: the whole history, and 2023-07 onward, after the three dated changes of 2023.

In [ ]:
# Given: annual counts for your line by discovery year, every status, as a table and as bars from zero.
line_ev = ev[ev["business_line"] == LINE]
annual = line_ev.groupby(line_ev["discovery_date"].dt.year.rename("year")).size().rename("events")
display(annual.to_frame().T)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(annual.index.astype(str), annual.values, color=OKABE_ITO[0])
for x_pos, value in zip(annual.index.astype(str), annual.values):
    ax.text(x_pos, value, f"{value:,}", ha="center", va="bottom")
ax.set_ylim(0, annual.max() * 1.15)
ax.set_xlabel("discovery year")
ax.set_ylabel("events recorded")
ax.set_title(f"{LINE}: events recorded by discovery year")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Figure: bar chart of the number of events your line recorded in each discovery year, with the count printed on each bar; the bars start at zero.

In [ ]:
# Given: monthly counts on the line's exposure months (zero months kept), and the dispersion check on two windows.
month_index = pd.PeriodIndex(exposure.loc[exposure["business_line"] == LINE, "month"], freq="M")
monthly = line_ev.groupby(line_ev["discovery_date"].dt.to_period("M")).size().reindex(month_index, fill_value=0)
windows = {"whole history": monthly, "2023-07 onward": monthly[monthly.index >= pd.Period("2023-07", freq="M")]}
freq_rows = []
for name, counts in windows.items():
    mean, var = counts.mean(), counts.var()
    dispersion_stat = ((counts - mean) ** 2).sum() / mean      # index of dispersion: chi-square with n - 1 degrees of freedom under Poisson
    p_value = stats.chi2.sf(dispersion_stat, len(counts) - 1)
    ll_poisson = stats.poisson.logpmf(counts, mean).sum()
    if var > mean:                                              # negative binomial by the method of moments; scipy's nbinom takes n and p, not a mean
        nb_n, nb_p = mean ** 2 / (var - mean), mean / var
        ll_nb = stats.nbinom.logpmf(counts, nb_n, nb_p).sum()
    else:                                                       # no overdispersion: the negative binomial collapses to the Poisson
        nb_n, nb_p, ll_nb = np.nan, np.nan, ll_poisson
    freq_rows.append({"window": name, "months": len(counts), "mean per month": round(mean, 2), "variance": round(var, 2),
                      "variance / mean": round(var / mean, 2), "dispersion test p-value": round(p_value, 4),
                      "log-likelihood, Poisson": round(ll_poisson, 1), "log-likelihood, negative binomial": round(ll_nb, 1),
                      "nbinom n": round(nb_n, 2), "nbinom p": round(nb_p, 3)})
freq_table = pd.DataFrame(freq_rows).set_index("window")
display(freq_table)

How to read it: the dispersion test's p-value is the chance of a ratio at least this far above 1 if the counts were Poisson; a small p-value says Poisson does not survive that window. The log-likelihood comparison says how much better the negative binomial describes the same months (higher is better; it has one more parameter, so a gain of 1 or 2 is nothing). One pitfall from the SciPy reference: `nbinom` is parameterized by `n` and `p`, not by a mean and a dispersion; the cell converts.

In [ ]:
# Given: monthly counts over the whole history, with the 2023-07 onward mean and the central 95% range each model allows around it.
recent = windows["2023-07 onward"]
mu = recent.mean()
pois_lo, pois_hi = stats.poisson.ppf([0.025, 0.975], mu)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(monthly.index.to_timestamp(), monthly.values, width=25, color=OKABE_ITO[5], label="events per month")
ax.axhline(mu, color=OKABE_ITO[7], linewidth=1.2, label=f"mean from 2023-07 ({mu:.1f})")
ax.axhspan(pois_lo, pois_hi, color=OKABE_ITO[0], alpha=0.15, label="Poisson: central 95%")
if recent.var() > mu:
    nb_lo, nb_hi = stats.nbinom.ppf([0.025, 0.975], mu ** 2 / (recent.var() - mu), mu / recent.var())
    ax.axhline(nb_lo, color=OKABE_ITO[3], linestyle="--", linewidth=1.2, label="negative binomial: central 95%")
    ax.axhline(nb_hi, color=OKABE_ITO[3], linestyle="--", linewidth=1.2)
ax.axvline(pd.Timestamp("2023-07-01"), color="gray", linestyle=":", linewidth=1)
ax.set_ylim(0, max(monthly.max(), pois_hi) * 1.15)
ax.set_xlabel("month of discovery")
ax.set_ylabel("events")
ax.set_title(f"{LINE}: monthly counts against the range each model allows")
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()
outside = int(((recent < pois_lo) | (recent > pois_hi)).sum())
print(f"months from 2023-07 outside the Poisson range: {outside} of {len(recent)} (about {0.05 * len(recent):.1f} expected if the counts were Poisson)")

Figure: bars of your line's monthly event counts over the whole history, a horizontal line at the mean from July 2023, a shaded band for the central 95% range a Poisson model allows, dashed lines for the wider negative binomial range when the counts are overdispersed, and a dotted vertical line at July 2023.

**Your frequency call.** Three lines, the same form as your Thursday position: the candidate you carry forward (Poisson or negative binomial), the evidence (which window, which ratio, which p-value), and the caveat (what would change the call: a regime, a threshold, a short history).

> Your answer:

### Step A3: How large do losses run?

Severity uses `Closed` rows only and `gross_loss`, which is complete on every closed row (`net_loss` is blank on some lines' early rows; that belongs in your fitness memo, not in the fit). Start with the shape, using the Module 4 views, then fit candidates.

In [ ]:
# Given: severity summary for your line, Closed rows only: center, spread, the upper quantiles, the largest event and the share of dollars in the top 1% of events.
sev = line_ev.loc[line_ev["status"] == "Closed", "gross_loss"].astype(float)
x = np.sort(sev.to_numpy())
k_top = int(np.ceil(0.01 * len(x)))
severity_summary = pd.Series({
    "closed events": len(x), "mean": x.mean(), "median": np.median(x), "standard deviation": x.std(ddof=1),
    "90th percentile": np.quantile(x, 0.90), "99th percentile": np.quantile(x, 0.99), "99.5th percentile": np.quantile(x, 0.995),
    "largest": x.max(), "share of dollars in the top 1% of events": x[-k_top:].sum() / x.sum()})
display(pd.DataFrame({LINE: [f"{v:,.3f}" if name.startswith("share") else f"{v:,.0f}" for name, v in severity_summary.items()]},
                     index=severity_summary.index))

In [ ]:
# Given: the Module 4 pair. The same closed gross losses on a linear axis (the view that hides the tail) and on a log axis with log-spaced bins.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(x, bins=50, color=OKABE_ITO[0])
axes[0].set_title("linear axis")
axes[0].set_xlabel("gross loss (dollars)")
axes[0].set_ylabel("closed events")
log_bins = np.logspace(np.log10(x.min()), np.log10(x.max()), 40)
axes[1].hist(x, bins=log_bins, color=OKABE_ITO[0])
axes[1].set_xscale("log")
axes[1].set_title("log axis, log-spaced bins")
axes[1].set_xlabel("gross loss (dollars, log scale)")
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"{LINE}: closed gross losses")
plt.tight_layout()
plt.show()

Figure: two histograms of your line's closed gross losses; on the left, a linear axis where almost every event sits in the first bar; on the right, a log axis with log-spaced bins showing the body as a hump and the tail stretching to the right.

In [ ]:
# Given: the ECDF on a log axis, with the median and the 99th percentile marked.
ecdf_y = np.arange(1, len(x) + 1) / len(x)
fig, ax = plt.subplots(figsize=(8, 4))
ax.step(x, ecdf_y, where="post", color=OKABE_ITO[0])
for q, style in [(0.5, "--"), (0.99, ":")]:
    ax.axvline(np.quantile(x, q), color=OKABE_ITO[7], linestyle=style, linewidth=1, label=f"{q:.0%} of events at or below {np.quantile(x, q):,.0f}")
ax.set_xscale("log")
ax.set_ylim(0, 1)
ax.set_xlabel("gross loss (dollars, log scale)")
ax.set_ylabel("share of closed events at or below")
ax.set_title(f"{LINE}: empirical cumulative distribution of closed gross losses")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Figure: a step curve rising from 0 to 1 showing the share of your line's closed events at or below each loss amount on a log axis, with a dashed vertical line at the median and a dotted one at the 99th percentile.

**Three candidates.** Each is fitted by maximum likelihood with SciPy's `fit`. SciPy writes every distribution with a location `loc`, a `scale` and, for most, a shape; losses start at zero, so the location is fixed at zero (`floc=0`) and the scale carries the size.

- **Lognormal**: the usual body model for losses; its tail is heavy but thins faster than a power law.
- **Log-logistic** (`fisk` in SciPy): fits a body like the lognormal and has a power-law tail, so it is the heavy-tailed alternative.
- **Gamma**: a light-tailed contrast; if it fits, the tail is not as heavy as it looks.

AIC (the Akaike information criterion) is minus twice the log-likelihood plus twice the number of fitted parameters; lower is better, and differences under about 2 are not meaningful.

In [ ]:
# Given: fit the three candidates with the location fixed at zero, and compare log-likelihood and AIC.
CANDIDATES = {"lognormal": stats.lognorm, "log-logistic": stats.fisk, "gamma": stats.gamma}
fits = {}
for name, dist in CANDIDATES.items():
    params = dist.fit(x, floc=0)                       # (shape, loc, scale); loc is held at 0
    loglik = dist.logpdf(x, *params).sum()
    fits[name] = {"dist": dist, "params": params, "loglik": loglik, "AIC": 2 * 2 - 2 * loglik}   # two free parameters each
fit_table = pd.DataFrame({name: {"shape": f["params"][0], "scale": f["params"][2], "log-likelihood": f["loglik"], "AIC": f["AIC"]}
                          for name, f in fits.items()}).T
display(fit_table.round({"shape": 3, "scale": 0, "log-likelihood": 1, "AIC": 1}))

In [ ]:
# Given: diagnostic view 1. The fitted densities drawn over the histogram of log10 losses, so the body and the tail are both visible.
log_x = np.log10(x)
grid = np.linspace(log_x.min(), log_x.max(), 400)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(log_x, bins=40, density=True, color="lightgray", label="closed events")
styles = {"lognormal": "-", "log-logistic": "--", "gamma": ":"}
for (name, f), color in zip(fits.items(), OKABE_ITO):
    density = f["dist"].pdf(10 ** grid, *f["params"]) * 10 ** grid * np.log(10)   # the density of log10(loss)
    ax.plot(grid, density, color=color, linestyle=styles[name], linewidth=1.8, label=name)
ax.set_xlim(log_x.min() - 0.1, log_x.max() + 0.1)
ticks = [t for t in np.arange(np.floor(log_x.min()), np.ceil(log_x.max()) + 1) if log_x.min() - 0.1 <= t <= log_x.max() + 0.1]
ax.set_xticks(ticks, [f"{10 ** t:,.0f}" for t in ticks])
ax.set_xlabel("gross loss (dollars, log scale)")
ax.set_ylabel("density of log10 loss")
ax.set_title(f"{LINE}: fitted densities over the closed losses")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Figure: a gray histogram of your line's closed losses on a log scale with three fitted density curves drawn over it: lognormal solid, log-logistic dashed, gamma dotted.

In [ ]:
# Given: diagnostic view 2. The fitted distribution functions over the ECDF, on a log axis; differences in the upper right are tail differences.
fig, ax = plt.subplots(figsize=(9, 4))
ax.step(x, ecdf_y, where="post", color="black", linewidth=1, label="empirical")
grid_x = np.logspace(np.log10(x.min()), np.log10(x.max()), 400)
for (name, f), color in zip(fits.items(), OKABE_ITO):
    ax.plot(grid_x, f["dist"].cdf(grid_x, *f["params"]), color=color, linestyle=styles[name], linewidth=1.8, label=name)
ax.set_xscale("log")
ax.set_ylim(0, 1.01)
ax.set_xlabel("gross loss (dollars, log scale)")
ax.set_ylabel("share at or below")
ax.set_title(f"{LINE}: fitted distribution functions against the ECDF")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Figure: the empirical cumulative distribution of your line's closed losses as a black step curve, with the three fitted distribution functions drawn over it in different line styles on a log axis.

In [ ]:
# Given: diagnostic view 3. Quantile comparison (probplot) against each candidate, on log axes; points above the line in the upper right mean the data's tail is heavier than the model's.
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, (name, f) in zip(axes, fits.items()):
    (model_q, data_q), _ = stats.probplot(x, sparams=f["params"], dist=f["dist"])
    ax.scatter(model_q, data_q, s=8, color=OKABE_ITO[0])
    lims = [min(model_q.min(), data_q.min()), max(model_q.max(), data_q.max())]
    ax.plot(lims, lims, color="black", linewidth=1)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(name)
    ax.set_xlabel("model quantile (dollars)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("observed quantile (dollars)")
fig.suptitle(f"{LINE}: quantile comparison against each candidate")
plt.tight_layout()
plt.show()

Figure: three scatter plots of observed against model quantiles on log axes, one per candidate, each with a diagonal reference line; points above the line at the upper right show where the observed tail is heavier than the model.

In [ ]:
# Given: the quantile table. Observed against each candidate at the median and the upper percentiles, in dollars and as a ratio to the observed value.
levels = [0.5, 0.9, 0.99, 0.995]
observed = pd.Series(np.quantile(x, levels), index=[f"{q:.1%}" for q in levels], name="observed")
quantile_table = pd.DataFrame({"observed": observed})
for name, f in fits.items():
    quantile_table[name] = f["dist"].ppf(levels, *f["params"])
ratios = quantile_table.drop(columns="observed").div(quantile_table["observed"], axis=0).add_suffix(" / observed")
display(quantile_table.map(lambda v: f"{v:,.0f}"))
display(ratios.round(2))

In [ ]:
# Optional: fit a fourth candidate of your choice the same way (for example stats.weibull_min with floc=0) and add it to the tables above.
# YOUR CODE HERE

**The comparison.** Fill in one row per candidate (add a row for any fourth candidate).

| Candidate | Fits the body? | Fits the tail? | Evidence (which view, which number) | Caveat |
|---|---|---|---|---|
| lognormal | | | | |
| log-logistic | | | | |
| gamma | | | | |

> Your answer:

**Your carry-forward choice.** The candidate you carry into Stage 2 and the caveats that travel with it: where it fails, by how much, and what that means for anyone using it to plan.

> Your answer:

### Step A4: Is the data fit for the purpose your charter states?

BCBS 239 was written for exactly this kind of data. Judge the register against four of its principles, for the purpose your charter states, not in general:

- **Principle 3, accuracy and integrity**: is the data correct and reconciled to its source; are there errors in values or structure?
- **Principle 4, completeness**: does the data capture every material event, for every period and field your purpose needs?
- **Principle 5, timeliness**: is the data current enough for the purpose, and do lags or open items distort the recent period?
- **Principle 6, adaptability**: can the data be re-cut for the questions the purpose will ask (by type, by period, by driver) without assumptions?

For each principle: the gap, where it is, how you found it, and its analytical consequence for your purpose. If you find no gap, write "no gap found" and name the check that supports it. The cell below collects the evidence for your line in one place.

In [ ]:
# Given: the evidence for the fitness memo, for your line: blanks, the open share, the reporting lag, the history span, the two eras and the rows after your preparation.
mine_raw = raw[raw["business_line"] == LINE]
mine_ev = ev[ev["business_line"] == LINE]
blanks_line = mine_ev.isna().sum()
print("blank cells by column (after your preparation):")
display(blanks_line[blanks_line > 0].rename("blank").to_frame().T if (blanks_line > 0).any() else pd.DataFrame({"blank": ["none"]}))
print(f"rows: {len(mine_raw):,} raw, {len(mine_ev):,} after your preparation")
print(f"open events: {int((mine_ev['status'] == 'Open').sum())} ({(mine_ev['status'] == 'Open').mean():.1%} of the line)")
print(f"discovered {mine_ev['discovery_date'].min():%Y-%m-%d} to {mine_ev['discovery_date'].max():%Y-%m-%d}")
months_per_year = mine_ev.groupby(mine_ev["discovery_date"].dt.year)["discovery_date"].apply(lambda s: s.dt.month.nunique())
print("years with fewer than 12 months of events:", [int(y) for y, m in months_per_year.items() if m < 12] or "none")
print("events by threshold_era:", mine_ev["threshold_era"].value_counts().to_dict())
_lag_line = (mine_ev["reporting_date"] - mine_ev["discovery_date"]).dt.days
print("median reporting lag (days) by discovery year:", _lag_line.groupby(mine_ev["discovery_date"].dt.year).median().to_dict())

**Purpose sentence, copied from your charter.**

> Your answer:

**Principle 3, accuracy and integrity.**

> Your answer:

**Principle 4, completeness.**

> Your answer:

**Principle 5, timeliness.**

> Your answer:

**Principle 6, adaptability.**

> Your answer:

### Step A5: The change line

On Thursday you posted a three-line position on the Desk Board: frequency call, leading severity candidate, top fitness gap. What changed since then, and on what evidence? Changing your position after reading your desk is permissible and professional, when attributed (name the desk-mate or the note). If nothing changed, write "nothing" and restate the position.

> Your answer:

## Part B: AI-assisted

About 1 hour. Part B grades your audit, not the AI's output. Points attach primarily to the critique memo and secondarily to the disposition table (decisions that hold up against your own Part A, reasons you can check). The other disclosure fields are graded for completion.

Expect fluent, confident proposals that ignore what the data cannot support. That is the failure mode this stage teaches you to catch.

Before starting, reread your week 1 AI Working Agreement and notice where this assignment sits relative to your own rules. You are not asked to submit anything about it, but the comparison is the beginning of the revision you will write in week 9.

### Step B1: What your assistant gets, and what it does not

Follow [the AI Protocol page](https://github.com/nybabot/ERM424/blob/main/docs/ai_protocol/ai_protocol.md). Give your assistant [the Module 5 role prompt](https://github.com/nybabot/ERM424/blob/main/docs/ai_protocol/role_prompt_m5.md) exactly as provided, then paste [the CRO sponsor letter](https://github.com/nybabot/ERM424/blob/main/docs/module_05/cro_sponsor_letter.md) in full, then make two asks:

1. Draft charter language for the CRO's mandate, field by field, on the template's fields.
2. Propose candidate frequency and severity models for your business line, by name.

Do not give it the data dictionary, the data, or anything from your Part A. The assistant works from a letter and a line name, which is exactly the situation the audit tests: what you know about this data is your advantage as the auditor.

### Step B2: Put the proposal on the page

Paste the assistant's two answers below, unedited, so the audit can point at them.

**The AI's charter language.**

> Paste the proposal here:

**The AI's frequency and severity proposals.**

> Paste the proposal here:

### Step B3: The disposition table

One row per element of the proposal: each charter field it drafted, each candidate model it proposed, and each assumption it stated or skipped. Decision: keep, revise, or reject, with a one-line reason grounded in your Part A (a check, a number, a dictionary fact). Add rows with `log_element(element, decision, reason)` and call `show_elements()` to see the table; for example `log_element("success criterion: 'fit a model with a p-value under 0.05'", "revise", "statistical, not decision-relevant: the CRO needs a dollar range and its uncertainty, which Stage 1 can show to be wrong")`.

In [ ]:
# Disposition table template: log_element(...) adds one row per element of the proposal; show_elements() displays the table.
DECISIONS = ["keep", "revise", "reject"]
element_dispositions = []

def log_element(element, decision, reason):
    """Add one row: an element of the AI's proposal, your decision on it, and a one-line reason."""
    if decision not in DECISIONS:
        print(f"Not added: decision must be one of {DECISIONS}, not {decision!r}.")
        return
    element_dispositions.append({"element": element, "decision": decision, "reason": reason})

def show_elements():
    """Display the disposition table."""
    if not element_dispositions:
        print("The table has no rows yet. Add one with log_element(...).")
        return
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(element_dispositions))

In [ ]:
# Your disposition rows: one log_element(...) call per element of the AI's proposal.
element_dispositions.clear()   # clears the table first, so running this cell again does not add every row twice
# YOUR CODE HERE

In [ ]:
# Given: check the disposition table. It lists blank reasons and counts rows by decision, and never stops the notebook.
if not element_dispositions:
    print("No rows to check yet.")
else:
    def _blank(value):
        """True for a missing entry: None, NaN, or text that is only spaces."""
        return value is None or (isinstance(value, float) and value != value) or not str(value).strip()
    blank_reasons = [i + 1 for i, row in enumerate(element_dispositions) if _blank(row["reason"])]
    print("rows with no reason:", blank_reasons if blank_reasons else "none")
    display(pd.Series([row["decision"] for row in element_dispositions]).value_counts().rename("rows").to_frame())

### Step B4: The critique memo

This is what is graded. At most 300 words. Identify at least two things the proposal assumed or ignored that the data cannot support, judged against your fitness memo, and at least one thing it did well or faster than you. For each item, say what you checked and how someone else could check it.

**Assumed or ignored, item 1.**

> Your answer:

**Assumed or ignored, item 2.**

> Your answer:

**Done well or faster.**

> Your answer:

### Step B5: The disclosure form

Complete every field. It should take about five minutes.

**Tool and access path.** Which AI assistant you used and how you reached it (for example, Colab's built-in assistance, or a named external tool through web chat).

> Your answer:

**Role prompt confirmation.** Confirm that you used the standard role prompt exactly as provided, followed by the sponsor letter. If you did not, say what differed.

> Your answer:

**Additional prompts.** Your own prompts and follow-up instructions beyond the role prompt, listed or pasted.

> Your answer:

**Additional context shared.** Anything you gave the assistant beyond the role prompt and the letter (for example, your line's name, column names, numbers from Part A).

> Your answer:

**Disposition of output.** For this module the disposition of output is the table you built in Step B3: one row per element, keep, revise or reject, with a one-line reason. The cell below shows it once more under this field.

In [ ]:
# Given: the disposition table, shown again as the disclosure form's disposition of output.
show_elements()

**Attestation.** One sentence confirming that the disclosure above is complete and accurate.

> Your answer:

## Deliverables checklist

Before you submit, confirm the following:

1. Your charter v1.0, on the template, as its own file beside this notebook, every field and both rationale fields completed, the amendment log started.
2. Part A in this notebook: the prep summary, the frequency call, the severity comparison and carry-forward choice, the fitness memo (purpose sentence and four principles), and the change line.
3. Part B in this notebook: the AI's proposal pasted, the disposition table, the critique memo, and the disclosure form with all six fields complete.
4. `LINE` set to your desk's line, and all cell outputs saved: run the notebook from the top before you share it.

## Submit

1. Save your copy to Google Drive if you have not already: File, then Save a copy in Drive.
2. Set the share link so your instructor can view it: Share, then under General access choose Anyone with the link, Viewer.
3. Paste the link into the Canvas assignment.

If anything fails, post in the Q&A discussion or bring it to office hours.